# Experiment 5.5.1.1 — Routing Diagnostics

**Analysis-only notebook.** It reads finalized Exp5.5.1.1 CSV/JSON artifacts and never runs diagnostics, retrains models, launches Slurm, or regenerates missing artifacts.

Scientific question: is Exp5.5.1 limited mainly by diffuse q or by weak expert specialization?

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Repository root not found')

REPO_ROOT = find_repo_root()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_5_5_1_1_routing_diagnostics' / 'routing_diagnostics_v1'
required = {
    'confidence': ROOT / 'confidence_metrics.csv',
    'confidence_summary': ROOT / 'confidence_summary.csv',
    'temperature': ROOT / 'temperature_sweep.csv',
    'temperature_summary': ROOT / 'temperature_summary.csv',
    'alignment': ROOT / 'temperature_alignment.csv',
    'weight': ROOT / 'expert_weight_similarity.csv',
    'functional': ROOT / 'expert_functional_diversity.csv',
    'expert_summary': ROOT / 'expert_summary.csv',
    'manifest': ROOT / 'manifest.json',
}
missing = [path for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Incomplete Exp5.5.1.1 finalization; missing: {missing}')
confidence = pd.read_csv(required['confidence'])
confidence_summary = pd.read_csv(required['confidence_summary'])
temperature = pd.read_csv(required['temperature'])
temperature_summary = pd.read_csv(required['temperature_summary'])
alignment = pd.read_csv(required['alignment'])
weight = pd.read_csv(required['weight'])
functional = pd.read_csv(required['functional'])
expert_summary = pd.read_csv(required['expert_summary'])
manifest = json.loads(required['manifest'].read_text())
manifest

## 1. q confidence
Inspect raw selected Exp5.5.1 q before any post-hoc sharpening.

In [ ]:
display(confidence.round(4))
display(confidence_summary.pivot(index='metric', columns='split', values='mean').round(4))

test_conf = confidence[confidence['split'] == 'test'].sort_values('seed')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(test_conf['seed'].astype(str), test_conf['mean_max_q'], marker='o', label='mean max(q)')
ax.plot(test_conf['seed'].astype(str), test_conf['mean_top1_top2_margin'], marker='o', label='mean top1-top2 margin')
ax.set_xlabel('Seed')
ax.set_ylabel('Probability')
ax.set_title('Selected Exp5.5.1 q confidence on test')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 2. Post-hoc routing sharpness
No temperature is selected here. The table/plots are diagnostic only.

In [ ]:
display(temperature_summary.round(4))
order = ['tau_1.00', 'tau_0.80', 'tau_0.60', 'tau_0.40', 'tau_0.25', 'hard_top1']
for split in ('val', 'test'):
    frame = temperature_summary[temperature_summary['split'] == split].set_index('variant').reindex(order)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.errorbar(order, frame['mean_balanced_accuracy'], yerr=frame['sem_balanced_accuracy'], marker='o', capsize=3)
    ax.set_xlabel('Post-hoc routing variant')
    ax.set_ylabel('Balanced accuracy')
    ax.set_title(f'{split}: routing sharpness vs BA')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.25)
    plt.show()

## 3. Does sharpening increase dependence on correct q/WHAT alignment?

In [ ]:
align_summary = alignment.groupby(['split', 'variant'], as_index=False)['ordered_minus_shuffled_balanced_accuracy'].agg(['mean', 'std']).reset_index()
display(align_summary.round(4))
test_align = align_summary[align_summary['split'] == 'test'].set_index('variant').reindex(order)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(order, test_align['mean'], marker='o')
ax.axhline(0.0, linewidth=1)
ax.set_xlabel('Post-hoc routing variant')
ax.set_ylabel('Ordered - shuffled q test BA')
ax.set_title('Temporal-alignment sensitivity')
ax.tick_params(axis='x', rotation=30)
ax.grid(alpha=0.25)
plt.show()

## 4. Expert specialization
Weight-space diversity and functional diversity are inspected separately.

In [ ]:
display(expert_summary.round(4))

mean_weight = weight.groupby(['expert_a', 'expert_b'])['weight_cosine_similarity'].mean()
matrix = np.eye(8)
for (a, b), value in mean_weight.items():
    matrix[int(a), int(b)] = value
    matrix[int(b), int(a)] = value
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, vmin=-1, vmax=1)
ax.set_xlabel('Expert')
ax.set_ylabel('Expert')
ax.set_title('Mean expert weight cosine similarity')
fig.colorbar(image, ax=ax)
plt.show()

test_func = functional[functional['split'] == 'test']
mean_func = test_func.groupby(['expert_a', 'expert_b'])['top_support_disagreement_rate'].mean()
matrix = np.zeros((8, 8))
for (a, b), value in mean_func.items():
    matrix[int(a), int(b)] = value
    matrix[int(b), int(a)] = value
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, vmin=0, vmax=1)
ax.set_xlabel('Expert')
ax.set_ylabel('Expert')
ax.set_title('Test: expert top-support disagreement')
fig.colorbar(image, ax=ax)
plt.show()

## Interpretation guide

- If lower temperature improves validation BA and increases Ordered-minus-shuffled while expert functional diversity is already substantial, the next experiment should target routing sharpness.
- If expert weight/evidence similarity is high and sharpening provides little benefit, the next experiment should target expert specialization or reduce K.
- If hard routing hurts while moderate sharpening helps, preserve soft transitions and use a controlled temperature rather than one-hot state assignment.
- Test results here remain post-hoc diagnostics; this notebook does not select a new deployable temperature or architecture.